# NSE Stocks + NIFTY 50 — Production Incremental Sync

## v6 — filesystem-first, terminal-failure aware, official NIFTY API

This notebook is designed to be safe to run repeatedly.

### Rules

1. **Parquet files are authoritative.**
2. If a Parquet exists, it is **never requested again**.
3. The manifest is diagnostic/state history, not the authority for successful files.
4. Previously failed **retryable** requests can be retried on a later run.
5. Known terminal cases such as NSE HTTP 404 are not retried forever.
6. Weekends are excluded from candidate dates.
7. NIFTY is downloaded through the official `niftyindices.com` historical API.
8. NIFTY is downloaded using **range/chunk API requests**, not one request per date.
9. NIFTY API responses are locally filtered because the endpoint can return dates outside the requested range.
10. Before every network request, the filesystem is checked again.
11. A NIFTY API/chunk failure does not create 4,100 permanent per-date failures.
12. A single NIFTY API test is run before the full sync.

**Important:** Do not mix cells from an older notebook with this one. Restart the Colab runtime and run this notebook top-to-bottom.


In [ ]:
# ============================================================
# 1. SETUP / IMPORTS
# ============================================================

import os
import io
import json
import time
import random
import zipfile
from pathlib import Path
from datetime import date, datetime, timedelta

import numpy as np
import pandas as pd
import requests

from tqdm.auto import tqdm

NOTEBOOK_VERSION = "NSE_STOCKS_NIFTY50_PRODUCTION_SYNC_v6"

print("=" * 80)
print(NOTEBOOK_VERSION)
print("=" * 80)
print("Started:", datetime.now().isoformat(timespec="seconds"))


In [ ]:
# ============================================================
# 2. MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

print("Google Drive mounted.")


In [ ]:
# ============================================================
# 3. CONFIGURATION
# ============================================================

BASE_DIR = Path("/content/drive/MyDrive/quant")

STOCK_DIR = BASE_DIR / "data" / "parquet"
NIFTY_DIR = BASE_DIR / "data" / "indices" / "nifty50"
MANIFEST_DIR = BASE_DIR / "data" / "manifests"

STOCK_MANIFEST_FILE = (
    MANIFEST_DIR / "nse_stock_download_manifest.csv"
)

NIFTY_MANIFEST_FILE = (
    MANIFEST_DIR / "nifty50_download_manifest.csv"
)

STOCK_DIR.mkdir(parents=True, exist_ok=True)
NIFTY_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

# Historical range.
START_DATE = date(2011, 1, 1)
END_DATE = date.today()

# Download switches.
DOWNLOAD_STOCKS = True
DOWNLOAD_NIFTY = True

# HTTP retry configuration.
REQUEST_TIMEOUT = 60
MAX_RETRIES = 4
RETRY_BASE_SECONDS = 2

# NIFTY official API range size.
NIFTY_CHUNK_DAYS = 90

# Only these statuses are permanently non-actionable.
# "error" remains retryable on a future run.
TERMINAL_STOCK_STATUSES = {
    "not_available",
    "no_data",
}

TERMINAL_NIFTY_STATUSES = {
    "no_data",
    "not_returned",
}

print("BASE_DIR             :", BASE_DIR)
print("STOCK_DIR            :", STOCK_DIR)
print("NIFTY_DIR            :", NIFTY_DIR)
print("MANIFEST_DIR         :", MANIFEST_DIR)
print("START_DATE           :", START_DATE)
print("END_DATE             :", END_DATE)
print("NIFTY_CHUNK_DAYS     :", NIFTY_CHUNK_DAYS)


In [ ]:
# ============================================================
# 4. COMMON HELPERS
# ============================================================

def trading_day_candidates(start_day, end_day):
    """Calendar dates excluding Saturday/Sunday."""

    current = start_day

    while current <= end_day:
        if current.weekday() < 5:
            yield current
        current += timedelta(days=1)


def parquet_path(directory, day):
    return Path(directory) / f"{day.isoformat()}.parquet"


def atomic_to_parquet(df, path):
    """Write Parquet atomically."""

    path = Path(path)
    tmp = path.with_suffix(".parquet.tmp")

    if tmp.exists():
        tmp.unlink()

    df.to_parquet(tmp, index=False)
    os.replace(tmp, path)


def append_manifest(row, manifest_path):
    """Append one diagnostic/state row to a CSV manifest."""

    row = dict(row)
    row.setdefault(
        "timestamp",
        datetime.now().isoformat(timespec="seconds"),
    )

    frame = pd.DataFrame([row])

    if manifest_path.exists():
        frame.to_csv(
            manifest_path,
            mode="a",
            header=False,
            index=False,
        )
    else:
        frame.to_csv(
            manifest_path,
            mode="w",
            header=True,
            index=False,
        )


def load_manifest(path):
    if not Path(path).exists():
        return pd.DataFrame()

    try:
        return pd.read_csv(path)
    except Exception as exc:
        print(
            "WARNING: unable to read manifest:",
            path,
            repr(exc),
        )
        return pd.DataFrame()


def latest_manifest_statuses(manifest):
    """
    Return the latest recorded status for each date.

    This is used ONLY to suppress known terminal dates.
    Filesystem existence always wins.
    """

    if manifest.empty:
        return {}

    required = {"date", "status"}

    if not required.issubset(manifest.columns):
        return {}

    work = manifest.copy()

    parsed_dates = pd.to_datetime(
        work["date"],
        errors="coerce",
    ).dt.date

    work["_date"] = parsed_dates
    work = work.dropna(subset=["_date"])

    # CSV append order represents chronological execution order.
    latest = {}

    for _, row in work.iterrows():
        latest[row["_date"]] = str(row["status"])

    return latest


In [ ]:
# ============================================================
# 5. HTTP SESSIONS
# ============================================================

NSE_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/153.0.0.0 Safari/537.36"
    ),
    "Accept": "*/*",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.nseindia.com/",
    "Connection": "keep-alive",
}

NIFTY_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/153.0.0.0 Safari/537.36"
    ),
    "Accept": (
        "application/json, text/javascript, */*; q=0.01"
    ),
    "Content-Type": "application/json; charset=utf-8",
    "Origin": "https://www.niftyindices.com",
    "Referer": "https://www.niftyindices.com/reports",
    "X-Requested-With": "XMLHttpRequest",
}


def create_nse_session():
    session = requests.Session()
    session.headers.update(NSE_HEADERS)

    try:
        response = session.get(
            "https://www.nseindia.com/",
            timeout=REQUEST_TIMEOUT,
        )
        print("NSE warm-up HTTP:", response.status_code)
    except Exception as exc:
        print("NSE warm-up warning:", repr(exc))

    return session


def create_nifty_session():
    session = requests.Session()
    session.headers.update(NIFTY_HEADERS)

    try:
        response = session.get(
            "https://www.niftyindices.com/reports",
            timeout=REQUEST_TIMEOUT,
        )
        print("NIFTY warm-up HTTP:", response.status_code)
    except Exception as exc:
        print("NIFTY warm-up warning:", repr(exc))

    return session


nse_session = create_nse_session()
nifty_session = create_nifty_session()


## STOCKS — NSE Bhavcopy

Legacy source is used through 2024-07-05.

UDiFF source is used from 2024-07-08 onward.

The stock queue is created from the filesystem first. Manifest terminal states are only used to avoid repeatedly requesting dates already confirmed unavailable.


In [ ]:
# ============================================================
# 6. NSE STOCK URL BUILDERS
# ============================================================

def legacy_stock_url(day):
    month = day.strftime("%b").upper()

    return (
        "https://nsearchives.nseindia.com/"
        "content/historical/EQUITIES/"
        f"{day.year}/{month}/"
        f"cm{day.strftime('%d')}{month}"
        f"{day.strftime('%Y')}bhav.csv.zip"
    )


def udiff_stock_url(day):
    return (
        "https://nsearchives.nseindia.com/"
        "content/cm/"
        f"BhavCopy_NSE_CM_0_0_0_"
        f"{day.strftime('%Y%m%d')}"
        "_F_0000.csv.zip"
    )


def stock_url(day):
    if day <= date(2024, 7, 5):
        return legacy_stock_url(day)

    return udiff_stock_url(day)


print("Legacy example:")
print(stock_url(date(2011, 1, 3)))

print()
print("Last legacy day:")
print(stock_url(date(2024, 7, 5)))

print()
print("First UDiFF day:")
print(stock_url(date(2024, 7, 8)))


In [ ]:
# ============================================================
# 7. STOCK BHA VCOPY PARSER
# ============================================================

def parse_stock_bhavcopy(content):
    with zipfile.ZipFile(io.BytesIO(content)) as z:
        csv_files = [
            name
            for name in z.namelist()
            if name.lower().endswith(".csv")
        ]

        if not csv_files:
            raise ValueError(
                f"No CSV inside ZIP. Files={z.namelist()}"
            )

        with z.open(csv_files[0]) as f:
            raw = pd.read_csv(f)

    raw.columns = [
        str(c).strip().upper()
        for c in raw.columns
    ]

    def find_column(candidates):
        for candidate in candidates:
            candidate = candidate.upper()
            if candidate in raw.columns:
                return candidate
        return None

    symbol_col = find_column([
        "SYMBOL",
        "TCKRSYMB",
        "TICKER",
    ])

    open_col = find_column([
        "OPEN",
        "OPNPRIC",
    ])

    high_col = find_column([
        "HIGH",
        "HGHPric".upper(),
        "HGHPRIC",
    ])

    low_col = find_column([
        "LOW",
        "LWPRIC",
    ])

    close_col = find_column([
        "CLOSE",
        "CLSPRIC",
    ])

    volume_col = find_column([
        "TOTTRDQTY",
        "TTL_TRD_QNTY",
        "TOTTRDQUANTITY",
        "TOTTRDQTY",
    ])

    columns = {
        "symbol": symbol_col,
        "open": open_col,
        "high": high_col,
        "low": low_col,
        "close": close_col,
        "volume": volume_col,
    }

    missing = [
        name
        for name, column in columns.items()
        if column is None
    ]

    if missing:
        raise ValueError(
            f"Missing stock fields={missing}; "
            f"available={list(raw.columns)}"
        )

    output = pd.DataFrame({
        "date": pd.NaT,
        "symbol": raw[symbol_col].astype(str).str.strip(),
        "open": pd.to_numeric(
            raw[open_col],
            errors="coerce",
        ),
        "high": pd.to_numeric(
            raw[high_col],
            errors="coerce",
        ),
        "low": pd.to_numeric(
            raw[low_col],
            errors="coerce",
        ),
        "close": pd.to_numeric(
            raw[close_col],
            errors="coerce",
        ),
        "volume": pd.to_numeric(
            raw[volume_col],
            errors="coerce",
        ),
    })

    output = output.dropna(
        subset=[
            "symbol",
            "open",
            "high",
            "low",
            "close",
        ]
    )

    return output


In [ ]:
# ============================================================
# 8. DOWNLOAD ONE STOCK DATE
# ============================================================

def download_stock_day(day):
    path = parquet_path(STOCK_DIR, day)
    url = stock_url(day)

    # --------------------------------------------------------
    # FILESYSTEM AUTHORITY — CHECK BEFORE ANY REQUEST
    # --------------------------------------------------------

    if path.exists():
        return {
            "date": day,
            "status": "skipped_existing",
            "rows": 0,
            "message": "",
            "path": str(path),
        }

    for attempt in range(1, MAX_RETRIES + 1):

        # ----------------------------------------------------
        # CHECK AGAIN IMMEDIATELY BEFORE HTTP
        # ----------------------------------------------------

        if path.exists():
            return {
                "date": day,
                "status": "skipped_existing",
                "rows": 0,
                "message": "",
                "path": str(path),
            }

        try:
            response = nse_session.get(
                url,
                timeout=REQUEST_TIMEOUT,
            )

            # A 404 means the requested archive does not exist.
            # This is terminal and will not be retried forever.
            if response.status_code == 404:
                return {
                    "date": day,
                    "status": "not_available",
                    "rows": 0,
                    "message": "HTTP 404",
                    "path": str(path),
                }

            response.raise_for_status()

            df = parse_stock_bhavcopy(
                response.content
            )

            if df.empty:
                return {
                    "date": day,
                    "status": "no_data",
                    "rows": 0,
                    "message": "Empty parsed dataframe",
                    "path": str(path),
                }

            df["date"] = pd.Timestamp(day)

            # ------------------------------------------------
            # FINAL CHECK BEFORE WRITE
            # ------------------------------------------------

            if path.exists():
                return {
                    "date": day,
                    "status": "skipped_existing",
                    "rows": 0,
                    "message": "",
                    "path": str(path),
                }

            atomic_to_parquet(
                df,
                path,
            )

            return {
                "date": day,
                "status": "downloaded",
                "rows": len(df),
                "message": "",
                "path": str(path),
            }

        except Exception as exc:

            if attempt >= MAX_RETRIES:
                return {
                    "date": day,
                    "status": "error",
                    "rows": 0,
                    "message": repr(exc),
                    "path": str(path),
                }

            sleep_seconds = (
                RETRY_BASE_SECONDS * (2 ** (attempt - 1))
                + random.uniform(0, 1)
            )

            time.sleep(sleep_seconds)


In [ ]:
# ============================================================
# 9. STOCK FILESYSTEM INVENTORY
# ============================================================

stock_candidates = list(
    trading_day_candidates(
        START_DATE,
        END_DATE,
    )
)

stock_existing = []
stock_missing = []

for day in stock_candidates:
    path = parquet_path(STOCK_DIR, day)

    if path.exists():
        stock_existing.append(day)
    else:
        stock_missing.append(day)

print("=" * 80)
print("STOCK FILESYSTEM INVENTORY")
print("=" * 80)

print("Candidate weekdays :", f"{len(stock_candidates):,}")
print("Existing Parquets  :", f"{len(stock_existing):,}")
print("Missing Parquets   :", f"{len(stock_missing):,}")


In [ ]:
# ============================================================
# 10. STOCK TERMINAL-FAILURE FILTER
# ============================================================

stock_manifest = load_manifest(
    STOCK_MANIFEST_FILE
)

latest_stock_status = latest_manifest_statuses(
    stock_manifest
)

stock_terminal_skip = []
stock_actionable_queue = []

for day in stock_missing:

    latest_status = latest_stock_status.get(day)

    if latest_status in TERMINAL_STOCK_STATUSES:
        stock_terminal_skip.append(day)
    else:
        stock_actionable_queue.append(day)

stock_download_queue = stock_actionable_queue

print("=" * 80)
print("STOCK QUEUE")
print("=" * 80)

print(
    "Known terminal dates skipped :",
    f"{len(stock_terminal_skip):,}",
)

print(
    "Actionable dates             :",
    f"{len(stock_download_queue):,}",
)

print()
print(
    "Terminal statuses:",
    TERMINAL_STOCK_STATUSES,
)


In [ ]:
# ============================================================
# 11. STOCK HARD PREFLIGHT
# ============================================================

existing_in_stock_queue = [
    day
    for day in stock_download_queue
    if parquet_path(
        STOCK_DIR,
        day,
    ).exists()
]

if existing_in_stock_queue:
    raise RuntimeError(
        "ABORT: stock queue contains existing files. "
        f"Examples={existing_in_stock_queue[:20]}"
    )

print("STOCK PREFLIGHT PASSED")
print(
    "Actionable queue:",
    len(stock_download_queue),
)


In [ ]:
# ============================================================
# 12. STOCK SYNC
# ============================================================

stock_started = datetime.now()
stock_results = []

if not DOWNLOAD_STOCKS:

    print("DOWNLOAD_STOCKS=False; skipping.")

elif not stock_download_queue:

    print("No actionable stock dates.")
    print("No stock HTTP requests will be made.")

else:

    print(
        f"Processing {len(stock_download_queue):,} "
        "actionable stock dates."
    )

    for day in tqdm(
        stock_download_queue,
        desc="NSE stock missing/retry dates",
    ):

        path = parquet_path(
            STOCK_DIR,
            day,
        )

        # Final filesystem guard.
        if path.exists():

            result = {
                "date": day,
                "status": "skipped_existing",
                "rows": 0,
                "message": "",
                "path": str(path),
            }

        else:

            result = download_stock_day(day)

        stock_results.append(result)

        append_manifest(
            result,
            STOCK_MANIFEST_FILE,
        )

stock_finished = datetime.now()

stock_results_df = pd.DataFrame(
    stock_results
)

print()
print("=" * 80)
print("STOCK SYNC COMPLETE")
print("=" * 80)
print("Started :", stock_started)
print("Finished:", stock_finished)
print("Elapsed :", stock_finished - stock_started)


In [ ]:
# ============================================================
# 13. STOCK SUMMARY
# ============================================================

if stock_results_df.empty:

    print("No stock download attempts were required.")

else:

    display(
        stock_results_df["status"]
        .value_counts()
        .rename_axis("status")
        .reset_index(name="count")
    )

    downloaded_rows = stock_results_df.loc[
        stock_results_df["status"] == "downloaded",
        "rows",
    ].sum()

    print(
        "Downloaded rows:",
        f"{downloaded_rows:,}",
    )

    errors = stock_results_df[
        stock_results_df["status"] == "error"
    ]

    if not errors.empty:
        print()
        print("Retryable stock errors:")
        display(
            errors[
                ["date", "message", "path"]
            ]
        )

    print()
    print(
        "Terminal/no-data dates this run:",
        int(
            stock_results_df["status"].isin(
                TERMINAL_STOCK_STATUSES
            ).sum()
        ),
    )


# NIFTY 50

The official endpoint is:

`https://www.niftyindices.com/BackPage/getHistoricaldatatabletoString`

The payload is the same structure used by the NIFTY Indices site's JavaScript:

```text
{
  "cinfo": "{'name':'NIFTY 50','startDate':'DD-MM-YYYY','endDate':'DD-MM-YYYY','indexName':'NIFTY 50'}"
}
```

The endpoint can return records outside the requested range. The notebook therefore filters every response locally before writing Parquet.

**There is no `sync_nifty50()` function in this notebook.**


In [ ]:
# ============================================================
# 14. NIFTY API CONFIGURATION
# ============================================================

NIFTY_ENDPOINT = (
    "https://www.niftyindices.com/"
    "BackPage/getHistoricaldatatabletoString"
)

print("NIFTY_ENDPOINT:")
print(NIFTY_ENDPOINT)


In [ ]:
# ============================================================
# 15. NIFTY PAYLOAD
# ============================================================

def make_nifty_payload(
    start_day,
    end_day,
):

    cinfo = (
        "{'name':'NIFTY 50',"
        f"'startDate':'{start_day.strftime('%d-%m-%Y')}',"
        f"'endDate':'{end_day.strftime('%d-%m-%Y')}',"
        "'indexName':'NIFTY 50'}"
    )

    return {
        "cinfo": cinfo
    }


print(
    json.dumps(
        make_nifty_payload(
            date(2025, 1, 1),
            date(2025, 1, 10),
        ),
        indent=2,
    )
)


In [ ]:
# ============================================================
# 16. NIFTY RESPONSE NORMALIZER
# ============================================================

def normalize_nifty_records(records):

    if not isinstance(records, list):
        raise ValueError(
            "Expected a JSON list from NIFTY API; "
            f"got {type(records).__name__}"
        )

    raw = pd.DataFrame(records)

    if raw.empty:
        return pd.DataFrame(
            columns=[
                "date",
                "symbol",
                "open",
                "high",
                "low",
                "close",
                "volume",
            ]
        )

    # The endpoint has used HistoricalDate in observed responses.
    # These aliases make the parser tolerant of minor field naming changes.
    normalized_names = {
        str(c).strip().lower(): c
        for c in raw.columns
    }

    def find_column(candidates):
        for candidate in candidates:
            found = normalized_names.get(
                candidate.lower()
            )
            if found is not None:
                return found
        return None

    date_col = find_column([
        "historicaldate",
        "historical date",
        "date",
        "indexdate",
    ])

    open_col = find_column([
        "open",
        "openprice",
        "open price",
    ])

    high_col = find_column([
        "high",
        "highprice",
        "high price",
    ])

    low_col = find_column([
        "low",
        "lowprice",
        "low price",
    ])

    close_col = find_column([
        "close",
        "closeprice",
        "closingindex",
        "closing index",
    ])

    missing = []

    if date_col is None:
        missing.append("date")

    if open_col is None:
        missing.append("open")

    if high_col is None:
        missing.append("high")

    if low_col is None:
        missing.append("low")

    if close_col is None:
        missing.append("close")

    if missing:
        raise ValueError(
            f"Could not identify NIFTY fields={missing}; "
            f"returned columns={list(raw.columns)}"
        )

    output = pd.DataFrame({
        "date": pd.to_datetime(
            raw[date_col],
            errors="coerce",
            dayfirst=True,
        ).dt.date,

        "symbol": "NIFTY50",

        "open": pd.to_numeric(
            raw[open_col],
            errors="coerce",
        ),

        "high": pd.to_numeric(
            raw[high_col],
            errors="coerce",
        ),

        "low": pd.to_numeric(
            raw[low_col],
            errors="coerce",
        ),

        "close": pd.to_numeric(
            raw[close_col],
            errors="coerce",
        ),

        "volume": np.nan,
    })

    output = output.dropna(
        subset=[
            "date",
            "open",
            "high",
            "low",
            "close",
        ]
    )

    return output


In [ ]:
# ============================================================
# 17. OFFICIAL NIFTY RANGE API
# ============================================================

def fetch_nifty_range(
    session,
    start_day,
    end_day,
):

    payload = make_nifty_payload(
        start_day,
        end_day,
    )

    for attempt in range(
        1,
        MAX_RETRIES + 1,
    ):

        try:

            response = session.post(
                NIFTY_ENDPOINT,
                headers=NIFTY_HEADERS,
                json=payload,
                timeout=REQUEST_TIMEOUT,
            )

            print()
            print(
                f"NIFTY API: {start_day} -> {end_day} | "
                f"attempt={attempt} | "
                f"HTTP={response.status_code} | "
                f"Content-Type={response.headers.get('Content-Type')} | "
                f"bytes={len(response.content):,}"
            )

            response.raise_for_status()

            # The endpoint may declare text/html while its body
            # is valid JSON. Parse the body itself.
            try:
                records = response.json()
            except Exception:
                records = json.loads(
                    response.text.strip()
                )

            df = normalize_nifty_records(
                records
            )

            if df.empty:
                print(
                    "NIFTY API returned no usable rows."
                )
                return df

            api_min = min(df["date"])
            api_max = max(df["date"])

            print(
                f"API actual range: "
                f"{api_min} -> {api_max}; "
                f"rows={len(df):,}"
            )

            # ------------------------------------------------
            # CRITICAL:
            # API can return records outside requested range.
            # Filter locally before writing anything.
            # ------------------------------------------------

            before = len(df)

            df = df[
                (df["date"] >= start_day)
                &
                (df["date"] <= end_day)
            ].copy()

            print(
                "Out-of-range rows removed:",
                before - len(df),
            )

            if not df.empty:
                assert min(df["date"]) >= start_day
                assert max(df["date"]) <= end_day

            return df

        except Exception as exc:

            print(
                "NIFTY API request failed:",
                repr(exc),
            )

            if attempt >= MAX_RETRIES:
                raise

            sleep_seconds = (
                RETRY_BASE_SECONDS * (2 ** (attempt - 1))
                + random.uniform(0, 1)
            )

            print(
                f"Retrying in {sleep_seconds:.1f}s..."
            )

            time.sleep(
                sleep_seconds
            )


In [ ]:
# ============================================================
# 18. NIFTY API TEST — RUN BEFORE FULL SYNC
# ============================================================

TEST_START = date(2025, 1, 1)
TEST_END = date(2025, 1, 10)

nifty_test_df = fetch_nifty_range(
    nifty_session,
    TEST_START,
    TEST_END,
)

print()
print("=" * 80)
print("NIFTY API TEST RESULT")
print("=" * 80)

if nifty_test_df.empty:

    print(
        "WARNING: no rows returned."
    )

else:

    print(
        "Requested range:",
        TEST_START,
        "->",
        TEST_END,
    )

    print(
        "Returned range:",
        min(nifty_test_df["date"]),
        "->",
        max(nifty_test_df["date"]),
    )

    print(
        "Rows:",
        len(nifty_test_df),
    )

    # Hard validation of the local filter.
    assert min(nifty_test_df["date"]) >= TEST_START
    assert max(nifty_test_df["date"]) <= TEST_END

    print()
    print("NIFTY API TEST PASSED")

    display(nifty_test_df)


In [ ]:
# ============================================================
# 19. NIFTY FILESYSTEM INVENTORY
# ============================================================

nifty_candidates = list(
    trading_day_candidates(
        START_DATE,
        END_DATE,
    )
)

nifty_existing = []
nifty_missing = []

for day in nifty_candidates:

    path = parquet_path(
        NIFTY_DIR,
        day,
    )

    if path.exists():
        nifty_existing.append(day)
    else:
        nifty_missing.append(day)

print("=" * 80)
print("NIFTY FILESYSTEM INVENTORY")
print("=" * 80)

print(
    "Candidate weekdays :",
    f"{len(nifty_candidates):,}",
)

print(
    "Existing Parquets  :",
    f"{len(nifty_existing):,}",
)

print(
    "Missing Parquets   :",
    f"{len(nifty_missing):,}",
)


In [ ]:
# ============================================================
# 20. NIFTY TERMINAL-FAILURE FILTER
# ============================================================

nifty_manifest = load_manifest(
    NIFTY_MANIFEST_FILE
)

latest_nifty_status = latest_manifest_statuses(
    nifty_manifest
)

nifty_terminal_skip = []
nifty_actionable_queue = []

for day in nifty_missing:

    latest_status = latest_nifty_status.get(day)

    if latest_status in TERMINAL_NIFTY_STATUSES:
        nifty_terminal_skip.append(day)
    else:
        nifty_actionable_queue.append(day)

nifty_download_queue = nifty_actionable_queue

print("=" * 80)
print("NIFTY ACTIONABLE QUEUE")
print("=" * 80)

print(
    "Known terminal dates skipped:",
    f"{len(nifty_terminal_skip):,}",
)

print(
    "Actionable missing dates:",
    f"{len(nifty_download_queue):,}",
)

print()
print(
    "Terminal statuses:",
    TERMINAL_NIFTY_STATUSES,
)


In [ ]:
# ============================================================
# 21. NIFTY HARD PREFLIGHT
# ============================================================

existing_in_nifty_queue = [
    day
    for day in nifty_download_queue
    if parquet_path(
        NIFTY_DIR,
        day,
    ).exists()
]

if existing_in_nifty_queue:
    raise RuntimeError(
        "ABORT: NIFTY queue contains existing files. "
        f"Examples={existing_in_nifty_queue[:20]}"
    )

print("NIFTY PREFLIGHT PASSED")
print(
    "Actionable dates:",
    len(nifty_download_queue),
)


In [ ]:
# ============================================================
# 22. NIFTY CHUNK PLANNER
# ============================================================

def make_nifty_chunks(
    missing_dates,
    chunk_days=90,
):

    if not missing_dates:
        return []

    start = min(missing_dates)
    end = max(missing_dates)

    chunks = []
    current = start

    while current <= end:

        chunk_end = min(
            current + timedelta(days=chunk_days - 1),
            end,
        )

        chunks.append(
            (current, chunk_end)
        )

        current = (
            chunk_end
            + timedelta(days=1)
        )

    return chunks


nifty_chunks = make_nifty_chunks(
    nifty_download_queue,
    NIFTY_CHUNK_DAYS,
)

print(
    "NIFTY API chunks:",
    len(nifty_chunks),
)

for index, (start, end) in enumerate(
    nifty_chunks[:20],
    start=1,
):
    print(
        f"{index:>3}: {start} -> {end}"
    )

if len(nifty_chunks) > 20:
    print("...")


In [ ]:
# ============================================================
# 23. SYNC NIFTY 50
# ============================================================
# THIS REPLACES THE OLD:
#
#     sync_nifty50(START_DATE, END_DATE, chunk_days=90)
#
# There is NO per-date API call here.
# One API call is made per date RANGE chunk.
# ============================================================

nifty_started = datetime.now()
nifty_results = []

if not DOWNLOAD_NIFTY:

    print(
        "DOWNLOAD_NIFTY=False; NIFTY sync skipped."
    )

elif not nifty_download_queue:

    print(
        "No actionable NIFTY dates."
    )

    print(
        "NO NIFTY API REQUESTS WILL BE MADE."
    )

else:

    print(
        f"Actionable NIFTY dates: "
        f"{len(nifty_download_queue):,}"
    )

    print(
        f"API chunks: "
        f"{len(nifty_chunks):,}"
    )

    missing_set = set(
        nifty_download_queue
    )

    for chunk_number, (
        chunk_start,
        chunk_end,
    ) in enumerate(
        nifty_chunks,
        start=1,
    ):

        chunk_missing = [
            day
            for day in nifty_download_queue
            if (
                chunk_start
                <= day
                <= chunk_end
            )
        ]

        if not chunk_missing:
            continue

        # ----------------------------------------------------
        # HARD CHECK BEFORE NETWORK REQUEST
        # ----------------------------------------------------

        existing_now = [
            day
            for day in chunk_missing
            if parquet_path(
                NIFTY_DIR,
                day,
            ).exists()
        ]

        if existing_now:
            raise RuntimeError(
                "ABORT: NIFTY files appeared "
                "after preflight and before API request. "
                f"Examples={existing_now[:20]}"
            )

        print()
        print("=" * 80)
        print(
            f"NIFTY CHUNK "
            f"{chunk_number}/{len(nifty_chunks)}"
        )
        print(
            "Range:",
            chunk_start,
            "->",
            chunk_end,
        )
        print(
            "Missing dates:",
            len(chunk_missing),
        )
        print("=" * 80)

        # ----------------------------------------------------
        # ONE OFFICIAL RANGE API REQUEST
        # ----------------------------------------------------

        try:

            df = fetch_nifty_range(
                nifty_session,
                chunk_start,
                chunk_end,
            )

        except Exception as exc:

            print(
                "NIFTY CHUNK ERROR:",
                repr(exc),
            )

            # IMPORTANT:
            # These remain retryable.
            # We do NOT mark them no_data/not_returned.
            for day in chunk_missing:

                result = {
                    "date": day,
                    "status": "error",
                    "rows": 0,
                    "message": repr(exc),
                    "path": str(
                        parquet_path(
                            NIFTY_DIR,
                            day,
                        )
                    ),
                    "chunk_start": chunk_start,
                    "chunk_end": chunk_end,
                }

                nifty_results.append(result)

                append_manifest(
                    result,
                    NIFTY_MANIFEST_FILE,
                )

            continue

        # ----------------------------------------------------
        # API returned no usable records
        # ----------------------------------------------------

        if df.empty:

            for day in chunk_missing:

                result = {
                    "date": day,
                    "status": "no_data",
                    "rows": 0,
                    "message": (
                        "No usable NIFTY records "
                        "returned for chunk"
                    ),
                    "path": str(
                        parquet_path(
                            NIFTY_DIR,
                            day,
                        )
                    ),
                    "chunk_start": chunk_start,
                    "chunk_end": chunk_end,
                }

                nifty_results.append(result)

                append_manifest(
                    result,
                    NIFTY_MANIFEST_FILE,
                )

            continue

        returned_dates = set(
            df["date"]
        )

        # ----------------------------------------------------
        # WRITE ONLY MISSING DATES
        # ----------------------------------------------------

        for day in chunk_missing:

            path = parquet_path(
                NIFTY_DIR,
                day,
            )

            # -----------------------------------------------
            # FINAL FILESYSTEM CHECK
            # -----------------------------------------------

            if path.exists():

                result = {
                    "date": day,
                    "status": "skipped_existing",
                    "rows": 0,
                    "message": "",
                    "path": str(path),
                    "chunk_start": chunk_start,
                    "chunk_end": chunk_end,
                }

                nifty_results.append(result)

                append_manifest(
                    result,
                    NIFTY_MANIFEST_FILE,
                )

                continue

            # -----------------------------------------------
            # API did not return this particular date.
            #
            # This is terminal for this data source unless
            # a future API behavior changes.
            # -----------------------------------------------

            if day not in returned_dates:

                result = {
                    "date": day,
                    "status": "not_returned",
                    "rows": 0,
                    "message": (
                        "Date absent from API response"
                    ),
                    "path": str(path),
                    "chunk_start": chunk_start,
                    "chunk_end": chunk_end,
                }

                nifty_results.append(result)

                append_manifest(
                    result,
                    NIFTY_MANIFEST_FILE,
                )

                continue

            day_df = df[
                df["date"] == day
            ].copy()

            if day_df.empty:

                result = {
                    "date": day,
                    "status": "not_returned",
                    "rows": 0,
                    "message": (
                        "Filtered dataframe empty"
                    ),
                    "path": str(path),
                    "chunk_start": chunk_start,
                    "chunk_end": chunk_end,
                }

                nifty_results.append(result)

                append_manifest(
                    result,
                    NIFTY_MANIFEST_FILE,
                )

                continue

            # Consistent Parquet date type.
            day_df["date"] = pd.Timestamp(day)

            # -----------------------------------------------
            # FINAL CHECK BEFORE WRITING
            # -----------------------------------------------

            if path.exists():

                status = "skipped_existing"
                rows = 0

            else:

                atomic_to_parquet(
                    day_df,
                    path,
                )

                status = "downloaded"
                rows = len(day_df)

            result = {
                "date": day,
                "status": status,
                "rows": rows,
                "message": "",
                "path": str(path),
                "chunk_start": chunk_start,
                "chunk_end": chunk_end,
            }

            nifty_results.append(result)

            append_manifest(
                result,
                NIFTY_MANIFEST_FILE,
            )

nifty_finished = datetime.now()

nifty_results_df = pd.DataFrame(
    nifty_results
)

print()
print("=" * 80)
print("NIFTY 50 SYNC COMPLETE")
print("=" * 80)
print("Started :", nifty_started)
print("Finished:", nifty_finished)
print("Elapsed :", nifty_finished - nifty_started)


In [ ]:
# ============================================================
# 24. NIFTY SUMMARY
# ============================================================

if nifty_results_df.empty:

    print(
        "No NIFTY download attempts were required."
    )

else:

    print("STATUS:")

    display(
        nifty_results_df["status"]
        .value_counts()
        .rename_axis("status")
        .reset_index(name="count")
    )

    downloaded_rows = nifty_results_df.loc[
        nifty_results_df["status"] == "downloaded",
        "rows",
    ].sum()

    print()
    print(
        "Downloaded rows:",
        f"{downloaded_rows:,}",
    )

    errors = nifty_results_df[
        nifty_results_df["status"] == "error"
    ]

    if not errors.empty:

        print()
        print(
            "RETRYABLE NIFTY ERRORS:"
        )

        display(
            errors[
                [
                    "date",
                    "message",
                    "path",
                    "chunk_start",
                    "chunk_end",
                ]
            ]
        )

    terminal = nifty_results_df[
        nifty_results_df["status"].isin(
            TERMINAL_NIFTY_STATUSES
        )
    ]

    print()
    print(
        "Terminal/no-data dates this run:",
        len(terminal),
    )


In [ ]:
# ============================================================
# 25. FINAL FILESYSTEM VALIDATION
# ============================================================

def validate_filesystem(
    directory,
    expected_dates,
    name,
):

    existing = []
    missing = []

    for day in expected_dates:

        path = parquet_path(
            directory,
            day,
        )

        if path.exists():
            existing.append(day)
        else:
            missing.append(day)

    print()
    print("=" * 80)
    print(name)
    print("=" * 80)

    print(
        "Expected dates:",
        f"{len(expected_dates):,}",
    )

    print(
        "Existing files:",
        f"{len(existing):,}",
    )

    print(
        "Still missing:",
        f"{len(missing):,}",
    )

    if missing:
        print()
        print(
            "First missing dates:"
        )
        print(
            missing[:50]
        )

    return missing


stock_final_missing = validate_filesystem(
    STOCK_DIR,
    stock_candidates,
    "STOCK FINAL FILESYSTEM VALIDATION",
)

nifty_final_missing = validate_filesystem(
    NIFTY_DIR,
    nifty_candidates,
    "NIFTY FINAL FILESYSTEM VALIDATION",
)


In [ ]:
# ============================================================
# 26. PARQUET SCHEMA VALIDATION
# ============================================================

EXPECTED_COLUMNS = [
    "date",
    "symbol",
    "open",
    "high",
    "low",
    "close",
    "volume",
]


def validate_sample_files(
    directory,
    sample_dates,
    name,
    sample_size=20,
):

    checked = 0

    for day in sample_dates:

        path = parquet_path(
            directory,
            day,
        )

        if not path.exists():
            continue

        df = pd.read_parquet(
            path
        )

        if list(df.columns) != EXPECTED_COLUMNS:

            raise AssertionError(
                f"{name} schema mismatch: "
                f"{path}
"
                f"Expected={EXPECTED_COLUMNS}
"
                f"Actual={list(df.columns)}"
            )

        if df.empty:

            raise AssertionError(
                f"{name} empty Parquet: {path}"
            )

        checked += 1

        if checked >= sample_size:
            break

    print(
        f"{name}: validated {checked} files."
    )


validate_sample_files(
    STOCK_DIR,
    stock_candidates,
    "STOCKS",
)

validate_sample_files(
    NIFTY_DIR,
    nifty_candidates,
    "NIFTY 50",
)


In [ ]:
# ============================================================
# 27. FINAL SUMMARY
# ============================================================

print()
print("=" * 90)
print("NSE STOCKS + NIFTY 50 PRODUCTION SYNC COMPLETE")
print("=" * 90)

print()
print("STOCKS")
print("-" * 50)
print(
    "Existing at start       :",
    f"{len(stock_existing):,}",
)
print(
    "Known terminal skipped  :",
    f"{len(stock_terminal_skip):,}",
)
print(
    "Actionable queue        :",
    f"{len(stock_download_queue):,}",
)
print(
    "Still missing           :",
    f"{len(stock_final_missing):,}",
)

print()
print("NIFTY 50")
print("-" * 50)
print(
    "Existing at start       :",
    f"{len(nifty_existing):,}",
)
print(
    "Known terminal skipped  :",
    f"{len(nifty_terminal_skip):,}",
)
print(
    "Actionable queue        :",
    f"{len(nifty_download_queue):,}",
)
print(
    "Still missing           :",
    f"{len(nifty_final_missing):,}",
)

print()
print("RULES")
print("-" * 50)
print("Existing Parquet        -> NEVER download")
print("Known terminal failure  -> DO NOT retry")
print("Retryable error         -> RETRY on future run")
print("Manifest                -> diagnostic/state history")
print("NIFTY                   -> official range API")
print("NIFTY                   -> local date filtering")
print("NIFTY                   -> NOT one API request/date")

print()
print("STOCK DIRECTORY:")
print(STOCK_DIR)

print()
print("NIFTY DIRECTORY:")
print(NIFTY_DIR)

print()
print("NOTEBOOK VERSION:")
print(NOTEBOOK_VERSION)
